# 03 — Robustness

Four checks, all from spec section 6. This is the section that separates a defensible
version of this project from a credulous one, because the central critique of this
literature (Durtschi & Easton 2005) is that the discontinuity can arise from
**scaling and sample-selection artifacts** rather than from manipulation.

- **6.1 Alternate denominators** — revenue and book equity on the SEC sample, market
  value of equity on the yfinance sample. If the notch survives every scaling choice,
  the artifact story weakens. If it vanishes under one, that is a real finding.
- **6.2 Size terciles** — the artifact story predicts the notch concentrates in small
  firms; the management story does not predict that as strongly.
- **6.3 Size-floor sensitivity** — does the result depend on including the smallest firms?
- **6.4 Cash-flow placebo** — the highest-value check. CFO is much harder to manage
  through accrual timing. A notch in net income but not in CFO supports the management
  interpretation; a notch in both points at the scaling.

In [1]:
import sys, warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

np.random.seed(20250811)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

DATA = ROOT / "data"
FIGS = ROOT / "figures"
FIGS.mkdir(exist_ok=True)
print("project root:", ROOT)

project root: /Users/riddhi/Downloads/All Of My Claude Skills/FinanceDSProject


In [2]:
from src.binning import BIN_WIDTHS
from src.panel import apply_filters, restrict_window
from src.discontinuity import run_all_widths, summarize
from src.plotting import placebo_panel, z_comparison

panel = pd.read_parquet(DATA / "panel.parquet")
SOURCE = "SEC Financial Statement Data Sets (10-K filings)"
window, _ = restrict_window(panel, "roa")
print(f"{len(panel):,} firm-years after filters; {len(window):,} inside the ROA window")

31,240 firm-years after filters; 16,018 inside the ROA window


## 6.1 Alternate denominators

Net income scaled by beginning-of-year total assets (headline), by revenue, and by
beginning-of-year book equity.

**Documented deviation from the spec:** the SEC flat files carry no share prices, so
market value of equity cannot be built on this sample. Book equity is used here as the
equity-based scalar, and market value of equity is tested separately below on the
yfinance sample, where price data exists. The substitution is stated rather than
quietly skipped.

In [3]:
denominators = [
    ("net income / lagged assets", "roa"),
    ("net income / revenue", "ni_revenue"),
    ("net income / lagged book equity", "ni_equity_lag"),
]

rows = []
for label, col in denominators:
    sub, info = restrict_window(panel, col)
    res = run_all_widths(sub[col], label=label)
    res["denominator"] = label
    res["n_with_measure"] = info["with_measure"]
    rows.append(res)

denom_results = pd.concat(rows, ignore_index=True)
denom_results[["denominator", "bin_width", "n_window", "count_below", "expected_below",
               "z_below", "p_below", "count_above", "expected_above", "z_above", "p_above"]].round(4)

,denominator,bin_width,n_window,count_below,expected_below,z_below,p_below,count_above,expected_above,z_above,p_above
0,net income / lagged assets,0.0025,16018,246,252.0,-0.3144,0.7532,291,252.5,1.9061,0.0566
1,net income / lagged assets,0.0050,16018,459,479.5,-0.7913,0.4288,550,496.0,1.9540,0.0507
2,net income / lagged assets,0.0100,16018,868,898.5,-0.8733,0.3825,1083,1017.0,1.7310,0.0834
3,net income / revenue,0.0025,13452,166,220.5,-3.3132,0.0009,254,220.0,1.8030,0.0714
4,net income / revenue,0.0050,13452,353,422.0,-2.9652,0.0030,528,406.0,4.6177,0.0000
5,net income / revenue,0.0100,13452,669,770.5,-3.2475,0.0012,987,852.0,3.7636,0.0002
6,net income / lagged book equity,0.0025,7177,88,100.5,-1.0728,0.2834,114,92.0,1.7557,0.0791
7,net income / lagged book equity,0.0050,7177,175,194.5,-1.2031,0.2289,210,182.5,1.6136,0.1066
8,net income / lagged book equity,0.0100,7177,354,345.5,0.3830,0.7018,400,384.5,0.6613,0.5084


In [4]:
for label, _ in denominators:
    print(f"--- {label}")
    print(summarize(denom_results[denom_results.denominator == label]))
    print()

--- net income / lagged assets
width=0.0025  N=16,018  below zero: 246 vs 252.0 expected (deficit, z=-0.31, p=0.753)  |  above zero: 291 vs 252.5 expected (surplus, z=+1.91, p=0.0566)
width=0.0050  N=16,018  below zero: 459 vs 479.5 expected (deficit, z=-0.79, p=0.429)  |  above zero: 550 vs 496.0 expected (surplus, z=+1.95, p=0.0507)
width=0.0100  N=16,018  below zero: 868 vs 898.5 expected (deficit, z=-0.87, p=0.383)  |  above zero: 1,083 vs 1017.0 expected (surplus, z=+1.73, p=0.0834)

--- net income / revenue
width=0.0025  N=13,452  below zero: 166 vs 220.5 expected (deficit, z=-3.31, p=0.000922)  |  above zero: 254 vs 220.0 expected (surplus, z=+1.80, p=0.0714)
width=0.0050  N=13,452  below zero: 353 vs 422.0 expected (deficit, z=-2.97, p=0.00303)  |  above zero: 528 vs 406.0 expected (surplus, z=+4.62, p=3.88e-06)
width=0.0100  N=13,452  below zero: 669 vs 770.5 expected (deficit, z=-3.25, p=0.00116)  |  above zero: 987 vs 852.0 expected (surplus, z=+3.76, p=0.000167)

--- net in

In [5]:
mid_width = denom_results[denom_results.bin_width == 0.005].copy()
fig = z_comparison(mid_width, label_col="denominator",
                   title="6.1  Does the notch survive a change of scalar?",
                   subtitle="Bin width 0.005. Source: " + SOURCE,
                   out=FIGS / "robustness_denominators.png")
fig

<Figure size 990x457.6 with 1 Axes>

### Market value of equity, on the yfinance sample

In [6]:
try:
    yf_panel = pd.read_parquet(DATA / "panel_yfinance.parquet")
    mve, mve_info = restrict_window(yf_panel, "ni_market_equity")
    print(mve_info)
    print(summarize(run_all_widths(mve["ni_market_equity"], label="NI / market equity")))
    print("\nThis sample is small; treat it as indicative only.")
except FileNotFoundError:
    print("yfinance panel not built; skipping the market-value denominator")

{'column': 'ni_market_equity', 'with_measure': 387, 'inside_window': 237, 'outside_window': 150, 'missing_measure': 14}


width=0.0025  N=237  below zero: 1 vs 4.0 expected (deficit, z=-1.75, p=0.0796)  |  above zero: 4 vs 1.5 expected (surplus, z=+1.16, p=0.247)
width=0.0050  N=237  below zero: 5 vs 4.5 expected (surplus, z=+0.19, p=0.851)  |  above zero: 6 vs 5.5 expected (surplus, z=+0.17, p=0.864)
width=0.0100  N=237  below zero: 8 vs 11.5 expected (deficit, z=-0.97, p=0.33)  |  above zero: 12 vs 21.0 expected (deficit, z=-2.01, p=0.0443)

This sample is small; treat it as indicative only.


## 6.2 Size terciles

Split on beginning-of-year total assets. If the notch is a small-firm phenomenon, the
scaling-artifact explanation gains ground.

In [7]:
window = window.copy()
window["size_tercile"] = pd.qcut(window["assets_lag"], 3,
                                 labels=["small", "medium", "large"])

rows = []
for tercile in ["small", "medium", "large"]:
    sub = window[window.size_tercile == tercile]
    res = run_all_widths(sub["roa"], label=tercile)
    res["tercile"] = tercile
    res["median_assets"] = sub["assets_lag"].median()
    rows.append(res)

size_results = pd.concat(rows, ignore_index=True)
size_results[["tercile", "median_assets", "bin_width", "n_window",
              "count_below", "expected_below", "z_below", "p_below",
              "count_above", "expected_above", "z_above", "p_above"]].round(4)

,tercile,median_assets,bin_width,n_window,count_below,expected_below,z_below,p_below,count_above,expected_above,z_above,p_above
0,small,1.957073e+08,0.0025,5340,100,93.5,0.5431,0.5871,111,95.0,1.2872,0.1980
1,small,1.957073e+08,0.0050,5340,176,180.0,-0.2509,0.8019,201,178.0,1.3832,0.1666
2,small,1.957073e+08,0.0100,5340,335,333.5,0.0699,0.9442,381,353.5,1.2211,0.2220
3,medium,1.548844e+09,0.0025,5339,79,90.5,-1.0431,0.2969,102,83.5,1.5608,0.1186
4,medium,1.548844e+09,0.0050,5339,158,162.0,-0.2641,0.7917,190,170.0,1.2337,0.2173
5,medium,1.548844e+09,0.0100,5339,292,301.0,-0.4447,0.6565,372,355.5,0.7378,0.4606
6,large,9.577625e+09,0.0025,5339,67,68.0,-0.1004,0.9201,78,74.0,0.3766,0.7065
7,large,9.577625e+09,0.0050,5339,125,137.5,-0.9134,0.3610,159,148.0,0.7347,0.4625
8,large,9.577625e+09,0.0100,5339,241,264.0,-1.2310,0.2183,330,308.0,1.0419,0.2974


In [8]:
fig = z_comparison(size_results[size_results.bin_width == 0.005],
                   label_col="tercile",
                   title="6.2  Is the notch concentrated in small firms?",
                   subtitle="Terciles of beginning-of-year total assets. Bin width 0.005.",
                   out=FIGS / "robustness_size_terciles.png")
fig

<Figure size 990x457.6 with 1 Axes>

## 6.3 Size-floor sensitivity

The headline sample uses a $10m floor on beginning-of-year assets. Tiny denominators
produce extreme scaled values and are the known source of the artifact, so the result
must be shown against the floor rather than at one convenient value.

Each floor is applied by re-running the full filter chain, so the drop accounting stays
correct at every threshold.

In [9]:
from src.panel import build_sec_panel
from src.build_panel import SEC_FISCAL_YEARS

raw = pd.read_parquet(DATA / "sec_raw_panel.parquet")   # cached, no download
base = build_sec_panel(raw)

rows = []
for floor in [0, 1e6, 10e6, 50e6, 100e6, 500e6]:
    clean, _ = apply_filters(base, size_floor=floor, fiscal_years=SEC_FISCAL_YEARS)
    sub, _ = restrict_window(clean, "roa")
    res = run_all_widths(sub["roa"], label=f"${floor:,.0f}")
    res["size_floor"] = f"assets >= ${floor/1e6:,.0f}m"
    res["n_panel"] = len(clean)
    rows.append(res)

floor_results = pd.concat(rows, ignore_index=True)
floor_results[["size_floor", "n_panel", "bin_width", "n_window",
               "count_below", "expected_below", "z_below",
               "count_above", "expected_above", "z_above"]].round(4)

,size_floor,n_panel,bin_width,n_window,count_below,expected_below,z_below,count_above,expected_above,z_above
0,assets >= $0m,39242,0.0025,16480,252,260.5,-0.4394,299,258.5,1.9791
1,assets >= $0m,39242,0.0050,16480,474,493.0,-0.7222,564,510.0,1.9287
2,assets >= $0m,39242,0.0100,16480,896,925.5,-0.8317,1110,1048.0,1.6048
3,assets >= $1m,34941,0.0025,16386,250,257.5,-0.3894,296,257.5,1.8891
4,assets >= $1m,34941,0.0050,16386,469,490.5,-0.8208,561,506.5,1.9522
5,assets >= $1m,34941,0.0100,16386,889,921.5,-0.9194,1105,1041.0,1.6609
6,assets >= $10m,31240,0.0025,16018,246,252.0,-0.3144,291,252.5,1.9061
7,assets >= $10m,31240,0.0050,16018,459,479.5,-0.7913,550,496.0,1.9540
8,assets >= $10m,31240,0.0100,16018,868,898.5,-0.8733,1083,1017.0,1.7310
9,assets >= $50m,26315,0.0025,14929,222,235.5,-0.7401,272,232.5,2.0272


In [10]:
fig = z_comparison(floor_results[floor_results.bin_width == 0.005],
                   label_col="size_floor",
                   title="6.3  Does the result depend on including the smallest firms?",
                   subtitle="Full filter chain re-run at each floor. Bin width 0.005.",
                   out=FIGS / "robustness_size_floor.png")
fig

<Figure size 990x629.2 with 1 Axes>

## 6.4 Cash-flow placebo — the decisive check

The same test, on cash flow from operations, scaled by the **same** beginning-of-year
total assets. CFO is far harder to shift across the zero line through accrual timing.

- Notch in net income but **not** in CFO → the anomaly is specific to accrual
  earnings, which is what the management interpretation predicts.
- Notch in **both** → suspect the scaling, not the managers. Whatever produces a
  break at zero is operating on a measure that cannot easily be managed.

Both measures are restricted to their own |x| <= 0.10 window and share a denominator,
so the comparison is like-for-like.

In [11]:
ni_win, ni_info = restrict_window(panel, "roa")
cfo_win, cfo_info = restrict_window(panel, "cfo_at")
print("net income:", ni_info)
print("cfo:       ", cfo_info)

placebo = pd.concat([
    run_all_widths(ni_win["roa"], label="net income / lagged assets").assign(measure="net income"),
    run_all_widths(cfo_win["cfo_at"], label="CFO / lagged assets").assign(measure="cash flow from operations"),
], ignore_index=True)

placebo[["measure", "bin_width", "n_window", "count_below", "expected_below", "z_below",
         "p_below", "count_above", "expected_above", "z_above", "p_above"]].round(4)

net income: {'column': 'roa', 'with_measure': 31240, 'inside_window': 16018, 'outside_window': 15222, 'missing_measure': 0}
cfo:        {'column': 'cfo_at', 'with_measure': 31199, 'inside_window': 13719, 'outside_window': 17480, 'missing_measure': 41}


,measure,bin_width,n_window,count_below,expected_below,z_below,p_below,count_above,expected_above,z_above,p_above
0,net income,0.0025,16018,246,252.0,-0.3144,0.7532,291,252.5,1.9061,0.0566
1,net income,0.0050,16018,459,479.5,-0.7913,0.4288,550,496.0,1.9540,0.0507
2,net income,0.0100,16018,868,898.5,-0.8733,0.3825,1083,1017.0,1.7310,0.0834
3,cash flow from operations,0.0025,13719,164,157.0,0.4530,0.6505,143,151.5,-0.5789,0.5627
4,cash flow from operations,0.0050,13719,335,291.0,2.0379,0.0416,282,319.0,-1.7879,0.0738
5,cash flow from operations,0.0100,13719,635,531.5,3.5484,0.0004,585,691.0,-3.5922,0.0003


In [12]:
print("--- net income")
print(summarize(placebo[placebo.measure == "net income"]))
print("\n--- cash flow from operations (placebo)")
print(summarize(placebo[placebo.measure == "cash flow from operations"]))

--- net income
width=0.0025  N=16,018  below zero: 246 vs 252.0 expected (deficit, z=-0.31, p=0.753)  |  above zero: 291 vs 252.5 expected (surplus, z=+1.91, p=0.0566)
width=0.0050  N=16,018  below zero: 459 vs 479.5 expected (deficit, z=-0.79, p=0.429)  |  above zero: 550 vs 496.0 expected (surplus, z=+1.95, p=0.0507)
width=0.0100  N=16,018  below zero: 868 vs 898.5 expected (deficit, z=-0.87, p=0.383)  |  above zero: 1,083 vs 1017.0 expected (surplus, z=+1.73, p=0.0834)

--- cash flow from operations (placebo)
width=0.0025  N=13,719  below zero: 164 vs 157.0 expected (surplus, z=+0.45, p=0.651)  |  above zero: 143 vs 151.5 expected (deficit, z=-0.58, p=0.563)
width=0.0050  N=13,719  below zero: 335 vs 291.0 expected (surplus, z=+2.04, p=0.0416)  |  above zero: 282 vs 319.0 expected (deficit, z=-1.79, p=0.0738)
width=0.0100  N=13,719  below zero: 635 vs 531.5 expected (surplus, z=+3.55, p=0.000388)  |  above zero: 585 vs 691.0 expected (deficit, z=-3.59, p=0.000328)


In [13]:
fig = placebo_panel(ni_win["roa"], cfo_win["cfo_at"], 0.005, source=SOURCE,
                    out=FIGS / "cfo_placebo.png")
fig

<Figure size 1276x528 with 2 Axes>

In [14]:
fig = z_comparison(placebo[placebo.bin_width == 0.005], label_col="measure",
                   title="6.4  Cash-flow placebo",
                   subtitle="Identical test, identical denominator. Bin width 0.005.",
                   out=FIGS / "robustness_placebo_z.png")
fig

<Figure size 990x400.4 with 1 Axes>

## Everything in one table

In [15]:
summary = pd.concat([
    denom_results.assign(check="6.1 denominator", spec=denom_results.denominator),
    size_results.assign(check="6.2 size tercile", spec=size_results.tercile),
    floor_results.assign(check="6.3 size floor", spec=floor_results.size_floor),
    placebo.assign(check="6.4 placebo", spec=placebo.measure),
], ignore_index=True)[["check", "spec", "bin_width", "n_window", "count_below",
                       "expected_below", "z_below", "p_below", "count_above",
                       "expected_above", "z_above", "p_above"]]

summary.to_csv(DATA / "robustness_results.csv", index=False)
print(f"wrote robustness_results.csv ({len(summary)} rows)")
summary.round(3)

wrote robustness_results.csv (42 rows)


,check,spec,bin_width,n_window,count_below,expected_below,z_below,p_below,count_above,expected_above,z_above,p_above
0,6.1 denominator,net income / lagged assets,0.002,16018,246,252.0,-0.314,0.753,291,252.5,1.906,0.057
1,6.1 denominator,net income / lagged assets,0.005,16018,459,479.5,-0.791,0.429,550,496.0,1.954,0.051
2,6.1 denominator,net income / lagged assets,0.010,16018,868,898.5,-0.873,0.383,1083,1017.0,1.731,0.083
3,6.1 denominator,net income / revenue,0.002,13452,166,220.5,-3.313,0.001,254,220.0,1.803,0.071
4,6.1 denominator,net income / revenue,0.005,13452,353,422.0,-2.965,0.003,528,406.0,4.618,0.000
5,6.1 denominator,net income / revenue,0.010,13452,669,770.5,-3.248,0.001,987,852.0,3.764,0.000
6,6.1 denominator,net income / lagged book equity,0.002,7177,88,100.5,-1.073,0.283,114,92.0,1.756,0.079
7,6.1 denominator,net income / lagged book equity,0.005,7177,175,194.5,-1.203,0.229,210,182.5,1.614,0.107
8,6.1 denominator,net income / lagged book equity,0.010,7177,354,345.5,0.383,0.702,400,384.5,0.661,0.508
9,6.2 size tercile,small,0.002,5340,100,93.5,0.543,0.587,111,95.0,1.287,0.198


## Reading the robustness table

The questions to answer from the numbers above, in order:

1. Does the sign and significance of `z_below` hold at **all three** bin widths?
2. Does it hold under **every** denominator, or only under lagged assets?
3. Is it concentrated in the **small** tercile — the pattern the artifact story predicts?
4. Does it survive raising the **size floor**?
5. Most important: is it **absent from CFO**? If CFO shows the same break, the
   scaling is the more likely explanation and the net-income result should not be
   read as evidence of management.

Whatever the answers, they get written up as they came out. A clean null, honestly
reported, is a stronger artifact than a p-hacked positive.